In [ ]:
# Stanford RNA 3D Folding 2 — Sequence-aware Template-Based Modeling (TBM)
# CPU-only, deterministic. Adapted from clean public TBM reference
# (analyticaobscura/rna-3d-folding-tbm) using the competition's own train data.
import subprocess, sys, glob, os

# Install biopython from offline wheel dataset (recursive find = mount-path agnostic)
try:
    import Bio  # may already be present in the image
except ImportError:
    _whls = glob.glob('/kaggle/input/**/biopython*.whl', recursive=True)
    print('biopython wheels found:', _whls)
    if not _whls:
        print('--- .whl files under /kaggle/input ---')
        for _r, _d, _f in os.walk('/kaggle/input'):
            for _fn in _f:
                if _fn.endswith('.whl'):
                    print(os.path.join(_r, _fn))
        raise FileNotFoundError('biopython wheel not found under /kaggle/input')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', _whls[0]], check=True)

import numpy as np
import pandas as pd
import random, hashlib, time, warnings
from pathlib import Path
from scipy.spatial import distance_matrix
from scipy.spatial.transform import Rotation as R
warnings.filterwarnings('ignore')
from Bio.Align import PairwiseAligner

SEED = 21
np.random.seed(SEED); random.seed(SEED)

# Global RNA-tuned aligner
ALIGNER = PairwiseAligner()
ALIGNER.mode = 'global'
ALIGNER.match_score = 2.5
ALIGNER.mismatch_score = -1
ALIGNER.open_gap_score = -10
ALIGNER.extend_gap_score = -0.5

# ---- Data path detection ----
ROOT = Path('/kaggle/input')
DATA_DIR = None
for p in ROOT.rglob('test_sequences.csv'):
    DATA_DIR = p.parent
    break
if DATA_DIR is None:
    raise FileNotFoundError('test_sequences.csv not found')
print('DATA_DIR:', DATA_DIR)

test_seqs = pd.read_csv(DATA_DIR / 'test_sequences.csv')
train_seqs = pd.read_csv(DATA_DIR / 'train_sequences.csv')
train_labels = pd.read_csv(DATA_DIR / 'train_labels.csv', low_memory=False)
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('train_seqs', train_seqs.shape, '| test_seqs', test_seqs.shape, '| sample_sub', sample_sub.shape)

SEQ_COL = 'sequence' if 'sequence' in test_seqs.columns else test_seqs.columns[1]
TID_COL = 'target_id' if 'target_id' in train_seqs.columns else train_seqs.columns[0]

# ---- Build train coord library ----
def process_labels(labels_df):
    df = labels_df.copy()
    df['target_id'] = df['ID'].str.rsplit('_', n=1).str[0]
    df = df.sort_values(['target_id', 'resid'])
    cc = ['x_1', 'y_1', 'z_1']
    arr = df[cc].values.astype(np.float64).copy()
    arr[arr < -1e6] = np.nan
    df[cc] = arr
    d = {}
    for tid, g in df.groupby('target_id', sort=False):
        d[tid] = g[cc].values
    return d

train_coords_dict = process_labels(train_labels)
print('train structures:', len(train_coords_dict))

# Keep only train seqs that have coords
train_seqs = train_seqs[train_seqs[TID_COL].isin(train_coords_dict)].reset_index(drop=True)
print('usable train templates:', len(train_seqs))

# ---- Alignment helpers ----
def get_alignment_score(query_seq, template_seq):
    al = ALIGNER.align(query_seq, template_seq)
    best = next(iter(al), None)
    if best is None:
        return 0.0
    max_possible = 2.1 * min(len(query_seq), len(template_seq))
    return min(best.score / max_possible, 1.0)

def build_aligned_sequences(query_seq, template_seq, alignment):
    qr, tr = alignment.aligned
    if len(qr) == 0:
        return None, None
    aq, at = [], []
    qp = tp = 0
    for (qs, qe), (ts, te) in zip(qr, tr):
        while qp < qs:
            aq.append(query_seq[qp]); at.append('-'); qp += 1
        while tp < ts:
            aq.append('-'); at.append(template_seq[tp]); tp += 1
        for i in range(qe - qs):
            aq.append(query_seq[qs + i]); at.append(template_seq[ts + i])
        qp, tp = qe, te
    while qp < len(query_seq):
        aq.append(query_seq[qp]); at.append('-'); qp += 1
    while tp < len(template_seq):
        aq.append('-'); at.append(template_seq[tp]); tp += 1
    return ''.join(aq), ''.join(at)

def get_aligned_sequences(query_seq, template_seq):
    al = ALIGNER.align(query_seq, template_seq)
    best = next(iter(al), None)
    if best is None:
        return None, None
    return build_aligned_sequences(query_seq, template_seq, best)

def find_similar_sequences(query_seq, top_n=5):
    out = []
    qlen = len(query_seq)
    for tid, tseq in TRAIN_SEQ_PAIRS:
        d = abs(len(tseq) - qlen) / max(len(tseq), qlen)
        if d > 0.5:
            continue
        s = get_alignment_score(query_seq, tseq)
        out.append((tid, tseq, s))
    out.sort(key=lambda x: x[2], reverse=True)
    return out[:top_n]

def fill_coordinate_gaps(coords):
    n = len(coords); step = 4.0
    for i in range(n):
        if np.isnan(coords[i, 0]):
            pv = next((j for j in range(i - 1, -1, -1) if not np.isnan(coords[j, 0])), -1)
            nv = next((j for j in range(i + 1, n) if not np.isnan(coords[j, 0])), -1)
            if pv >= 0 and nv >= 0:
                w = (i - pv) / (nv - pv)
                coords[i] = (1 - w) * coords[pv] + w * coords[nv]
    for i in range(n):
        if np.isnan(coords[i, 0]):
            if i == 0:
                fv = next((j for j in range(1, n) if not np.isnan(coords[j, 0])), -1)
                if fv >= 0:
                    for j in range(fv - 1, -1, -1):
                        dd = np.random.normal(0, 1, 3)
                        dd = dd / (np.linalg.norm(dd) + 1e-10) * step
                        coords[j] = coords[j + 1] - dd
                else:
                    coords = generate_basic_structure_coords(n); break
            else:
                pv = next((j for j in range(i - 1, -1, -1) if not np.isnan(coords[j, 0])), -1)
                if pv >= 0:
                    dd = np.random.normal(0, 1, 3)
                    dd = dd / (np.linalg.norm(dd) + 1e-10) * step
                    coords[i] = coords[pv] + dd
    return np.nan_to_num(coords)

def generate_basic_structure_coords(n):
    c = np.zeros((n, 3))
    for i in range(n):
        a = i * 0.6
        c[i] = [10.0 * np.cos(a), 10.0 * np.sin(a), i * 2.5]
    return c

def adapt_template_simple(query_seq, template_seq, tcoords):
    qc = np.zeros((len(query_seq), 3))
    scale = len(tcoords) / len(query_seq)
    for i in range(len(query_seq)):
        ti = min(int(i * scale), len(tcoords) - 1)
        qc[i] = tcoords[ti] if not np.any(np.isnan(tcoords[ti])) else [np.nan, np.nan, np.nan]
    return fill_coordinate_gaps(qc)

def adapt_template_to_query(query_seq, template_seq, tcoords):
    aq, at = get_aligned_sequences(query_seq, template_seq)
    if aq is None:
        return adapt_template_simple(query_seq, template_seq, tcoords)
    qc = np.full((len(query_seq), 3), np.nan)
    qi = ti = 0
    for i in range(len(aq)):
        qch, tch = aq[i], at[i]
        if qch != '-' and tch != '-':
            if qi < len(query_seq) and ti < len(tcoords) and not np.any(np.isnan(tcoords[ti])):
                qc[qi] = tcoords[ti]
            ti += 1; qi += 1
        elif qch != '-' and tch == '-':
            qi += 1
        elif qch == '-' and tch != '-':
            ti += 1
    return fill_coordinate_gaps(qc)

def adaptive_rna_constraints(coords, sequence, confidence=1.0):
    rc = coords.copy(); n = len(sequence)
    cs = 0.8 * (1.0 - min(confidence, 0.8))
    lo, hi = 5.5, 6.5
    for i in range(n - 1):
        cur, nxt = rc[i], rc[i + 1]
        dist = np.linalg.norm(nxt - cur)
        if dist < lo or dist > hi:
            tgt = (lo + hi) / 2
            dr = nxt - cur
            dr = dr / (np.linalg.norm(dr) + 1e-10)
            adj = (tgt - dist) * cs
            rc[i + 1] = cur + dr * (dist + adj)
    mind = 3.8
    dm = distance_matrix(rc, rc)
    cl = np.where((dm < mind) & (dm > 0))
    for k in range(len(cl[0])):
        i, j = cl[0][k], cl[1][k]
        if abs(i - j) <= 1 or i >= j:
            continue
        pi, pj = rc[i], rc[j]
        dr = pj - pi
        dr = dr / (np.linalg.norm(dr) + 1e-10)
        adj = (mind - dm[i, j]) * cs
        rc[i] = pi - dr * (adj / 2)
        rc[j] = pj + dr * (adj / 2)
    return rc

def generate_rna_structure(sequence, seed=None):
    if seed is not None:
        np.random.seed(seed); random.seed(seed)
    n = len(sequence)
    c = np.zeros((n, 3))
    for i in range(min(3, n)):
        a = i * 0.6
        c[i] = [10.0 * np.cos(a), 10.0 * np.sin(a), i * 2.5]
    cd = np.array([0.0, 0.0, 1.0])
    comp = {'G': 'C', 'C': 'G', 'A': 'U', 'U': 'A'}
    for i in range(3, n):
        base = sequence[i]; paired = False; pidx = -1
        w = min(i, 15)
        for j in range(i - w, i):
            if j >= 0 and sequence[j] == comp.get(base, 'X'):
                paired = True; pidx = j; break
        if paired and i - pidx <= 10 and random.random() < 0.7:
            pp = c[pidx]
            off = np.random.normal(0, 1, 3) * 2.0
            bpd = 10.0 + random.uniform(-1.0, 1.0)
            center = np.mean(c[:i], axis=0)
            dr = center - pp
            dr = dr / (np.linalg.norm(dr) + 1e-10)
            c[i] = pp + dr * bpd + off
            cd = np.random.normal(0, 0.3, 3)
            cd = cd / (np.linalg.norm(cd) + 1e-10)
        else:
            if random.random() < 0.3:
                ang = random.uniform(0.2, 0.6)
                ax = np.random.normal(0, 1, 3)
                ax = ax / (np.linalg.norm(ax) + 1e-10)
                cd = R.from_rotvec(ang * ax).apply(cd)
            else:
                cd += np.random.normal(0, 0.15, 3)
                cd = cd / (np.linalg.norm(cd) + 1e-10)
            c[i] = c[i - 1] + random.uniform(3.5, 4.5) * cd
    return c

def predict_rna_structures(sequence, target_id, n_predictions=5):
    preds = []
    sims = find_similar_sequences(sequence, top_n=n_predictions)
    for tid, tseq, sim in sims:
        ac = adapt_template_to_query(sequence, tseq, train_coords_dict[tid])
        rc = adaptive_rna_constraints(ac, sequence, confidence=sim)
        scale = max(0.05, 0.8 - sim)
        preds.append(rc + np.random.normal(0, scale, rc.shape))
        if len(preds) >= n_predictions:
            break
    while len(preds) < n_predictions:
        us = f"{target_id}_{len(preds)}"
        sv = int(hashlib.sha256(us.encode()).hexdigest(), 16) % (2**32 - 1)
        dn = generate_rna_structure(sequence, seed=sv)
        preds.append(adaptive_rna_constraints(dn, sequence, confidence=0.2))
    return preds[:n_predictions]

# ---- Build predictions aligned to sample_submission ----
TRAIN_SEQ_PAIRS = list(zip(train_seqs[TID_COL].astype(str), train_seqs['sequence'].astype(str)))
test_seq_map = dict(zip(test_seqs[TID_COL if TID_COL in test_seqs.columns else 'target_id'].astype(str),
                        test_seqs[SEQ_COL].astype(str)))

submission = sample_sub.copy()
submission['_target'] = submission['ID'].str.rsplit('_', n=1).str[0]

t0 = time.time()
groups = list(submission.groupby('_target', sort=False))
print('test targets:', len(groups))
for gi, (target_id, group) in enumerate(groups):
    idx = group.index
    seq = test_seq_map.get(str(target_id))
    if seq is None or len(seq) != len(group):
        seq = 'A' * len(group)
    rs = int(hashlib.sha256(str(target_id).encode()).hexdigest(), 16) % (2**32 - 1)
    np.random.seed(rs); random.seed(rs)
    preds = predict_rna_structures(seq, str(target_id), n_predictions=5)
    for s in range(5):
        submission.loc[idx, f'x_{s+1}'] = np.round(preds[s][:, 0], 3)
        submission.loc[idx, f'y_{s+1}'] = np.round(preds[s][:, 1], 3)
        submission.loc[idx, f'z_{s+1}'] = np.round(preds[s][:, 2], 3)
    if (gi + 1) % 10 == 0:
        print(f'  {gi+1}/{len(groups)} targets done, {time.time()-t0:.0f}s')

submission = submission.drop(columns=['_target'])
out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)
print('Saved', out_path, submission.shape)
print(submission.head())
